# 05 — RT-DETR + Multi-Tracker Evaluation

Applies ByteTrack / BotSort (via **boxmot**) to the RT-DETR detector from `04b_train_rtdetr.ipynb`.

Prerequisites: `01_prepare_dataset.ipynb`, `04b_train_rtdetr.ipynb`

## 0 — Configuration

In [ ]:
from pathlib import Path
import json, itertools, random
import cv2, numpy as np, torch

REPO_ROOT    = Path('../')
MOT_DIR      = REPO_ROOT / 'data/mot_dataset'
RTDETR_RUNS  = (REPO_ROOT / 'runs/rtdetr').resolve()
TRACKER_RUNS = (REPO_ROOT / 'runs/trackers_rtdetr').resolve()
TRACKER_RUNS.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR   = REPO_ROOT / 'data/processed/labelstudiochunks'
CONF = 0.20;  IOU = 0.7
TRACKERS = ['bytetrack.yaml', 'botsort.yaml']
HP_SEARCH_SUBSET = 0.4
HP_GRID = {
    'det_conf':          [0.15, 0.25, 0.35],
    'det_iou':           [0.5,  0.7],
    'track_high_thresh': [0.25, 0.40],
    'track_low_thresh':  [0.10, 0.20],
    'new_track_thresh':  [0.30, 0.45],
    'track_buffer':      [20,   40],
    'match_thresh':      [0.7,  0.85],
    'pp_cc_iou':         [None, 0.4, 0.6],
    'pp_merge_time':     [None, 20, 40],
    'pp_merge_dist':     [20.0, 40.0],
    'pp_merge_interp':   [True, False],
}
HP_MAX_TRIALS = 8
with open(REPO_ROOT / 'data/splits/splits.json') as f:
    splits = json.load(f)
CLASSES = splits['classes']
FINAL_RTDETR_WEIGHTS = RTDETR_RUNS / 'final' / 'weights' / 'best.pt'
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Config OK')
print(f'  RT-DETR weights: {FINAL_RTDETR_WEIGHTS}')
print(f'  Device: {DEVICE}')

## 1 — Install boxmot

In [ ]:
import subprocess, sys
try:
    import boxmot; print('boxmot ready')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'boxmot', '-q'], check=True)
    import boxmot; print('boxmot installed')

## 2 — Helpers

In [ ]:
from ultralytics import RTDETR
import pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mpatches

_PALETTE = [(255,56,56),(255,157,151),(255,112,31),(255,178,29),(207,210,49),
            (72,249,10),(146,204,23),(61,219,134),(26,147,52),(0,212,187)]

def class_color(cid):
    return _PALETTE[int(cid) % len(_PALETTE)]

def draw_tracks(frame, boxes, track_ids, class_ids, class_names, confidences=None):
    out = frame.copy()
    for i, (box, tid, cid) in enumerate(zip(boxes, track_ids, class_ids)):
        x1, y1, x2, y2 = map(int, box)
        color = class_color(cid)[::-1]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        label = f"{class_names[int(cid)]} #{int(tid)}"
        if confidences is not None:
            label += f" {confidences[i]:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(out, (x1, y1 - th - 4), (x1 + tw, y1), color, -1)
        cv2.putText(out, label, (x1, y1 - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return out

CROSS_CLASS_NMS = True
CROSS_CLASS_IOU_THRESH = 0.5
AUTO_MERGE_TRACKS = True
MERGE_TIME_THRESHOLD = 30
MERGE_DISTANCE_THRESH = 30.0

def _iou(b1, b2):
    ax1, ay1 = b1[0], b1[1]; ax2, ay2 = b1[0] + b1[2], b1[1] + b1[3]
    bx1, by1 = b2[0], b2[1]; bx2, by2 = b2[0] + b2[2], b2[1] + b2[3]
    inter = max(0, min(ax2, bx2) - max(ax1, bx1)) * max(0, min(ay2, by2) - max(ay1, by1))
    union = b1[2] * b1[3] + b2[2] * b2[3] - inter
    return inter / union if union > 0 else 0.0

def _cx(b): return b[0] + b[2] / 2
def _cy(b): return b[1] + b[3] / 2

def _cross_class_nms(dets, cc_iou_thresh=None):
    if cc_iou_thresh is None or len(dets) < 2:
        return dets
    dets = sorted(dets, key=lambda d: -d[6])
    keep = [True] * len(dets)
    for i in range(len(dets)):
        if not keep[i]: continue
        for j in range(i + 1, len(dets)):
            if not keep[j] or dets[i][7] == dets[j][7]: continue
            if _iou(dets[i][2:6], dets[j][2:6]) >= cc_iou_thresh:
                keep[j] = False
    return [d for d, k in zip(dets, keep) if k]

def _merge_track_rows(rows, merge_time=None, merge_dist=30.0, interpolate=True):
    if merge_time is None: return rows
    from collections import defaultdict
    tracks = defaultdict(list)
    for r in rows: tracks[r[1]].append(r)
    for tid in tracks: tracks[tid].sort(key=lambda r: r[0])
    meta = [{"tid": tid, "cid": s[0][7], "start": s[0][0], "end": s[-1][0],
             "rows": s, "merged_into": None} for tid, s in tracks.items()]
    meta.sort(key=lambda m: m["start"])
    merged = 0
    for j in range(len(meta)):
        if meta[j]["merged_into"] is not None: continue
        cands = []
        for i in range(j):
            if meta[i]["merged_into"] is not None or meta[i]["cid"] != meta[j]["cid"]: continue
            gap = meta[j]["start"] - meta[i]["end"]
            if gap < 0 or gap > merge_time: continue
            last = meta[i]["rows"][-1]; first = meta[j]["rows"][0]
            dist = ((_cx(last[2:6]) - _cx(first[2:6]))**2 + (_cy(last[2:6]) - _cy(first[2:6]))**2)**0.5
            dp = dist / max(last[4], last[5], first[4], first[5], 1) * 100
            if dp <= merge_dist: cands.append((i, dp, gap))
        if cands:
            bi = sorted(cands, key=lambda c: (c[1], c[2]))[0][0]
            gap_rows = []
            if interpolate:
                re = meta[bi]["rows"][-1]; rs = meta[j]["rows"][0]
                f0, f1 = int(re[0]), int(rs[0])
                for f in range(f0 + 1, f1):
                    t = (f - f0) / (f1 - f0)
                    gap_rows.append([f, meta[j]["tid"],
                                     re[2] + t*(rs[2]-re[2]), re[3] + t*(rs[3]-re[3]),
                                     re[4] + t*(rs[4]-re[4]), re[5] + t*(rs[5]-re[5]),
                                     0.0, meta[j]["cid"]])
            meta[j]["rows"] = meta[bi]["rows"] + gap_rows + meta[j]["rows"]
            meta[j]["start"] = meta[bi]["start"]
            meta[bi]["merged_into"] = meta[j]["tid"]
            merged += 1
    if merged: print(f"    [merge] merged {merged} track(s)")
    out = []
    for m in meta:
        if m["merged_into"] is None: out.extend(m["rows"])
    return out

def tracker_name(t): return Path(t).stem

def compute_mot_metrics(gt_dir, pred_dir):
    try: import motmetrics as mm
    except ImportError: return {"MOTA": None, "IDF1": None, "ID_Sw": None}
    if not hasattr(np, 'asfarray'):
        np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)
    accs, seq_names = [], []
    for gt_file in sorted(Path(gt_dir).glob("*/gt/gt.txt")):
        seq = gt_file.parent.parent.name
        pred_file = Path(pred_dir) / f"{seq}.txt"
        if not pred_file.exists(): continue
        gt = np.loadtxt(gt_file, delimiter=",")
        pred = np.loadtxt(pred_file, delimiter=",") if pred_file.stat().st_size > 0 else np.zeros((0, 8))
        if gt.ndim == 1: gt = gt.reshape(1, -1)
        if pred.ndim == 1: pred = pred.reshape(1, -1)
        if gt.shape[0] == 0: continue
        acc = mm.MOTAccumulator(auto_id=True)
        for fn in sorted(set(gt[:, 0].astype(int))):
            gt_fn = gt[gt[:, 0] == fn]
            pr_fn = pred[pred[:, 0] == fn] if len(pred) else np.zeros((0, 8))
            if len(gt_fn) and len(pr_fn):
                dist = mm.distances.iou_matrix(gt_fn[:, 2:6], pr_fn[:, 2:6], max_iou=0.5)
            else:
                dist = np.full((len(gt_fn), max(len(pr_fn), 0)), np.nan)
            acc.update(gt_fn[:, 1].astype(int).tolist(),
                       pr_fn[:, 1].astype(int).tolist() if len(pr_fn) else [], dist)
        accs.append(acc); seq_names.append(seq)
    if not accs: return {"MOTA": None, "IDF1": None, "ID_Sw": None}
    mh = mm.metrics.create()
    summary = mh.compute_many(accs, metrics=["mota", "idf1", "num_switches"],
                               names=seq_names, generate_overall=True)
    row = summary.loc["OVERALL"]
    return {"MOTA": float(row["mota"]), "IDF1": float(row["idf1"]), "ID_Sw": int(row["num_switches"])}

_MODEL_CACHE = {}

def _get_rtdetr(weights_path):
    k = str(weights_path)
    if k not in _MODEL_CACHE:
        _MODEL_CACHE[k] = RTDETR(k)
    return _MODEL_CACHE[k]

def _detect_rtdetr(frame_bgr, model, conf=0.20, iou=0.7):
    "Run RT-DETR on a single BGR frame. Returns (N,6) array: [x1,y1,x2,y2,score,class_id]."
    results = model.predict(frame_bgr, conf=conf, iou=iou, verbose=False, device=DEVICE)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return np.zeros((0, 6), dtype=np.float32)
    xyxy  = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy().reshape(-1, 1)
    clss  = boxes.cls.cpu().numpy().reshape(-1, 1)
    return np.concatenate([xyxy, confs, clss], axis=1).astype(np.float32)

def _make_tracker(yaml_name, hp):
    from boxmot import ByteTrack, BotSort
    kw = dict(track_high_thresh=hp.get('track_high_thresh', 0.25),
              track_low_thresh=hp.get('track_low_thresh', 0.10),
              new_track_thresh=hp.get('new_track_thresh', 0.30),
              track_buffer=hp.get('track_buffer', 30),
              match_thresh=hp.get('match_thresh', 0.8),
              frame_rate=30)
    name = Path(yaml_name).stem.lower()
    return ByteTrack(**kw) if 'bytetrack' in name else BotSort(**kw)

_TRACKER_KEYS = ['det_conf', 'det_iou', 'track_high_thresh', 'track_low_thresh',
                 'new_track_thresh', 'track_buffer', 'match_thresh']
_PP_KEYS = ['pp_cc_iou', 'pp_merge_time', 'pp_merge_dist', 'pp_merge_interp']

def _count_gt_tracks(gt_dir, seq_names):
    for seq in seq_names:
        gtf = Path(gt_dir) / 'test' / seq / 'gt' / 'gt.txt'
        if not gtf.exists(): continue
        gt = np.loadtxt(gtf, delimiter=',')
        if gt.ndim == 1: gt = gt.reshape(1, -1)
        print(f"    [GT] {seq}: {len(set(gt[:, 1].astype(int)))} unique tracks")

def run_tracker_rtdetr(weights_path, video_paths, output_dir, tracker_yaml,
                       hp_overrides=None, pp_overrides=None, conf=CONF):
    pp = pp_overrides or {}
    cc_iou       = pp.get('pp_cc_iou',     CROSS_CLASS_IOU_THRESH if CROSS_CLASS_NMS else None)
    merge_time   = pp.get('pp_merge_time',  MERGE_TIME_THRESHOLD if AUTO_MERGE_TRACKS else None)
    merge_dist   = pp.get('pp_merge_dist',  MERGE_DISTANCE_THRESH)
    merge_interp = pp.get('pp_merge_interp', True)
    hp = dict(hp_overrides or {})
    run_conf = hp.pop('det_conf', conf)
    run_iou  = hp.pop('det_iou', IOU)

    model = _get_rtdetr(str(weights_path))
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    _count_gt_tracks(MOT_DIR, [Path(vp).stem for vp in video_paths if Path(vp).exists()])
    all_preds = {}
    for vp in video_paths:
        vp = Path(vp)
        if not vp.exists():
            print(f"  [SKIP] {vp.name}"); continue
        trk = _make_tracker(tracker_yaml, hp)
        cap = cv2.VideoCapture(str(vp)); rows = []; fi = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            fi += 1
            dets = _detect_rtdetr(frame, model, conf=run_conf, iou=run_iou)
            tracks = trk.update(dets, frame)
            frows = []
            for t in tracks:
                x1, y1, x2, y2 = t[0], t[1], t[2], t[3]
                tid = int(t[4])
                cf  = float(t[5]) if len(t) > 5 else 1.0
                cid = int(t[6])   if len(t) > 6 else 0
                frows.append([fi, tid, x1, y1, x2 - x1, y2 - y1, cf, cid])
            frows = _cross_class_nms(frows, cc_iou_thresh=cc_iou)
            rows.extend(frows)
        cap.release()
        rows = _merge_track_rows(rows, merge_time=merge_time,
                                  merge_dist=merge_dist, interpolate=merge_interp)
        (output_dir / f"{vp.stem}.txt").write_text("
".join(
            f"{r[0]},{r[1]},{r[2]:.2f},{r[3]:.2f},{r[4]:.2f},{r[5]:.2f},{r[6]:.4f},{r[7]}"
            for r in rows))
        all_preds[vp.stem] = rows
        print(f"  {vp.name}: {len(rows)} detections")
    return all_preds

print('Helpers ready.')

## 3 — Hyperparameter Search

In [ ]:
if not FINAL_RTDETR_WEIGHTS.exists():
    raise FileNotFoundError(f'RT-DETR weights not found: {FINAL_RTDETR_WEIGHTS}')

test_chunk_names = splits['mot_test_chunks']
all_test_videos = sorted([CHUNKS_DIR / f for f in test_chunk_names if (CHUNKS_DIR / f).exists()])
print(f'Total test videos: {len(all_test_videos)}')
random.seed(42)
n_sub = max(1, int(len(all_test_videos) * HP_SEARCH_SUBSET))
search_videos = random.sample(all_test_videos, n_sub)
print(f'HP search subset: {len(search_videos)}/{len(all_test_videos)}')

tg = {k: HP_GRID[k] for k in _TRACKER_KEYS if k in HP_GRID}
pg = {k: HP_GRID[k] for k in _PP_KEYS if k in HP_GRID}
all_combos = [dict(**dict(zip(tg, tc)), **dict(zip(pg, pc)))
              for tc, pc in itertools.product(itertools.product(*tg.values()),
                                               itertools.product(*pg.values()))]
random.shuffle(all_combos)
combos = all_combos[:HP_MAX_TRIALS]
print(f'HP trials: {len(combos)}/{len(all_combos)} total')

best_configs = {}; best_metrics = {}; hp_search_log = []
for tracker in TRACKERS:
    tname = tracker_name(tracker)
    print(f'\n{"="*60}\nHP search — {tname}\n{"="*60}')
    best_mota = -float('inf')
    for trial_idx, combo in enumerate(combos):
        t_hp   = {k: v for k, v in combo.items() if k in _TRACKER_KEYS}
        p_hp   = {k: v for k, v in combo.items() if k in _PP_KEYS}
        det_hp = {k: t_hp[k] for k in ('det_conf', 'det_iou') if k in t_hp}
        print(f'  Trial {trial_idx+1}/{len(combos)}')
        print(f'    detection: {det_hp}')
        print(f'    tracker  : { {k:v for k,v in t_hp.items() if k not in det_hp} }')
        print(f'    post-proc: {p_hp}')
        pred_dir = TRACKER_RUNS / tname / 'hp_search' / f'trial_{trial_idx:03d}'
        run_tracker_rtdetr(FINAL_RTDETR_WEIGHTS, search_videos, pred_dir, tracker,
                           hp_overrides=t_hp, pp_overrides=p_hp)
        m = compute_mot_metrics(MOT_DIR / 'test', pred_dir)
        print(f"    -> MOTA={m['MOTA']}  IDF1={m['IDF1']}  ID_Sw={m['ID_Sw']}")
        hp_search_log.append({'tracker': tname, 'trial': trial_idx, **combo, **m})
        if m['MOTA'] is not None and m['MOTA'] > best_mota:
            best_mota = m['MOTA']
            best_configs[tracker] = combo
            best_metrics[tracker] = m
    print(f'  Best MOTA={best_mota:.4f}')

pd.DataFrame(hp_search_log).to_csv(TRACKER_RUNS / 'hp_search_log.csv', index=False)
for t, hp in best_configs.items():
    print(f'  {tracker_name(t):20s}: {hp}')

## 4 — Final Evaluation

In [ ]:
final_results = {}
eval_videos = search_videos  # swap to all_test_videos for full eval
for tracker in TRACKERS:
    tname   = tracker_name(tracker)
    best_hp = best_configs.get(tracker, {})
    t_hp    = {k: v for k, v in best_hp.items() if k in _TRACKER_KEYS}
    p_hp    = {k: v for k, v in best_hp.items() if k in _PP_KEYS}
    pred_dir = TRACKER_RUNS / tname / 'final_eval'
    print(f'Running {tname} ({len(eval_videos)} videos)...')
    run_tracker_rtdetr(FINAL_RTDETR_WEIGHTS, eval_videos, pred_dir, tracker,
                       hp_overrides=t_hp, pp_overrides=p_hp)
    m = compute_mot_metrics(MOT_DIR / 'test', pred_dir)
    final_results[tname] = {'metrics': m, 'hp': best_hp, 'pred_dir': str(pred_dir)}
    print(f"  -> MOTA={m['MOTA']}  IDF1={m['IDF1']}  ID_Sw={m['ID_Sw']}")

df_final = pd.DataFrame([{'tracker': k, **v['metrics']} for k, v in final_results.items()])
print(df_final.round(4).to_string(index=False))
df_final.to_csv(TRACKER_RUNS / 'test_metrics.csv', index=False)
with open(TRACKER_RUNS / 'test_metrics.json', 'w') as f:
    json.dump({k: v['metrics'] for k, v in final_results.items()}, f, indent=2)
with open(TRACKER_RUNS / 'best_configs.json', 'w') as f:
    json.dump({k: v['hp'] for k, v in final_results.items()}, f, indent=2)

## 4b — Per-Class Count Metrics

In [ ]:
import os as _os

def _count_tracks_per_class(txt_file, n_classes, col_cid=7, col_tid=1):
    counts = np.zeros(n_classes, dtype=int)
    if not Path(txt_file).exists() or Path(txt_file).stat().st_size == 0:
        return counts
    data = np.loadtxt(txt_file, delimiter=',')
    if data.ndim == 1: data = data.reshape(1, -1)
    for cid in range(n_classes):
        mask = data[:, col_cid].astype(int) == cid
        counts[cid] = len(set(data[mask, col_tid].astype(int))) if mask.any() else 0
    return counts

def compute_count_metrics(gt_dir, pred_dir, classes):
    n = len(classes); ga, pa = [], []
    for seq in sorted(_os.listdir(Path(gt_dir))):
        gtf = Path(gt_dir) / seq / 'gt' / 'gt.txt'
        pf  = Path(pred_dir) / f'{seq}.txt'
        if not gtf.exists(): continue
        gd = np.loadtxt(gtf, delimiter=',')
        if gd.ndim == 1: gd = gd.reshape(1, -1)
        if gd.shape[0] == 0 or gd.shape[1] < 8: continue
        gc = np.zeros(n, dtype=int)
        for i in range(n):
            mask = gd[:, 7].astype(int) == (i + 1)
            gc[i] = len(set(gd[mask, 1].astype(int))) if mask.any() else 0
        ga.append(gc); pa.append(_count_tracks_per_class(pf, n))
    if not ga: return pd.DataFrame()
    ga = np.array(ga, dtype=float); pa = np.array(pa, dtype=float)
    gm = ga.mean(0); pm = pa.mean(0)
    ce = (pa - ga).mean(0); ace = np.abs(pa - ga).mean(0)
    denom = np.maximum(pa, ga)
    ca = np.where(denom > 0, np.minimum(pa, ga) / denom, 1.0).mean(0)
    rows = [{'class': c, 'GT_mean': round(gm[i], 2), 'Pred_mean': round(pm[i], 2),
             'CE': round(ce[i], 3), 'ACE': round(ace[i], 3), 'CA': round(ca[i], 3)}
            for i, c in enumerate(classes)]
    rows.append({'class': 'MACRO_AVG', 'GT_mean': round(gm.mean(), 2),
                 'Pred_mean': round(pm.mean(), 2), 'CE': round(ce.mean(), 3),
                 'ACE': round(ace.mean(), 3), 'CA': round(ca.mean(), 3)})
    return pd.DataFrame(rows)

count_results = {}
for tracker in TRACKERS:
    tname    = tracker_name(tracker)
    pred_dir = Path(final_results[tname]['pred_dir'])
    df_cnt   = compute_count_metrics(MOT_DIR / 'test', pred_dir, CLASSES)
    count_results[tname] = df_cnt
    print(df_cnt.to_string(index=False))

with open(TRACKER_RUNS / 'count_metrics.json', 'w') as f:
    json.dump({t: df.to_dict(orient='records') for t, df in count_results.items()}, f, indent=2)

## 5 — Metric Visualizations

In [ ]:
tnames = [tracker_name(t) for t in TRACKERS]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#4C72B0', '#DD8452']

for ax, metric in zip(axes, ['MOTA', 'IDF1', 'ID_Sw']):
    vals = [final_results[n]['metrics'].get(metric) or 0 for n in tnames]
    bars = ax.bar(tnames, vals, color=colors[:len(tnames)])
    ax.set_title(metric)
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}' if metric != 'ID_Sw' else str(int(val)),
                ha='center', va='bottom', fontsize=10)

plt.suptitle('RT-DETR Tracker Comparison (best HP)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(TRACKER_RUNS / 'metric_barplot.png', dpi=150, bbox_inches='tight')
plt.show()

import seaborn as sns
df_hp_log = pd.DataFrame(hp_search_log)
if 'track_high_thresh' in HP_GRID and 'match_thresh' in HP_GRID and not df_hp_log.empty:
    fig, axes = plt.subplots(1, len(TRACKERS), figsize=(7 * len(TRACKERS), 5))
    if len(TRACKERS) == 1: axes = [axes]
    for ax, tracker in zip(axes, TRACKERS):
        name = tracker_name(tracker)
        sub  = df_hp_log[df_hp_log['tracker'] == name].copy()
        if sub.empty or sub['MOTA'].isna().all():
            ax.set_title(f'{name} — no data'); continue
        pivot = sub.pivot_table(index='track_high_thresh', columns='match_thresh',
                                 values='MOTA', aggfunc='mean')
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax,
                    linewidths=0.5, cbar_kws={'label': 'MOTA'})
        ax.set_title(f'{name} — MOTA (track_high_thresh x match_thresh)')
    plt.tight_layout()
    plt.savefig(TRACKER_RUNS / 'hp_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {TRACKER_RUNS}/hp_heatmap.png')

## 5b — Per-Class Count Metrics Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RT-DETR Per-Class Count Metrics — Test Set', fontweight='bold')
bar_colors = ['#4C72B0', '#DD8452']

for ax, metric, ylabel in zip(axes,
                                ['CE', 'ACE', 'CA'],
                                ['Count Error (CE)', 'Abs Count Error (ACE)', 'Count Accuracy (CA)']):
    class_labels = CLASSES
    x     = np.arange(len(class_labels))
    width = 0.8 / len(TRACKERS)
    for ti, tracker in enumerate(TRACKERS):
        tname = tracker_name(tracker)
        df    = count_results[tname]
        vals  = [df.loc[df['class'] == c, metric].values[0]
                 if c in df['class'].values else 0 for c in class_labels]
        offset = (ti - len(TRACKERS) / 2 + 0.5) * width
        ax.bar(x + offset, vals, width, label=tname,
               color=bar_colors[ti % len(bar_colors)], alpha=0.85)
    if metric == 'CE':
        ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_xticks(x); ax.set_xticklabels(class_labels, rotation=30, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(ylabel)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
out_path = TRACKER_RUNS / 'count_metrics.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 6 — Annotated Frame Snapshots

Extract 12 evenly-spaced frames from the first eval video with predicted tracks overlaid.

In [ ]:
import math

demo_video = eval_videos[0] if eval_videos else None

def annotated_grid_rtdetr(video_path, weights_path, tracker_yaml, best_hp, n_frames=12):
    hp = dict(best_hp)
    run_conf = hp.pop('det_conf', CONF)
    run_iou  = hp.pop('det_iou', IOU)
    t_hp = {k: v for k, v in hp.items()
            if k in _TRACKER_KEYS and k not in ('det_conf', 'det_iou')}
    model = _get_rtdetr(str(weights_path))
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    trk   = _make_tracker(tracker_yaml, t_hp)
    frames_data = []
    cap = cv2.VideoCapture(str(video_path)); fi = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        fi += 1
        dets   = _detect_rtdetr(frame, model, conf=run_conf, iou=run_iou)
        tracks = trk.update(dets, frame)
        frames_data.append((frame.copy(), tracks))
    cap.release()
    indices = [int(i * (total - 1) / (n_frames - 1)) for i in range(n_frames)]
    out_frames = []
    for fi in indices:
        frame, tracks = frames_data[min(fi, len(frames_data) - 1)]
        boxes, tids, cids, confs = [], [], [], []
        for t in tracks:
            boxes.append(t[:4]); tids.append(int(t[4]))
            confs.append(float(t[5]) if len(t) > 5 else 1.0)
            cids.append(int(t[6]) if len(t) > 6 else 0)
        out_frames.append(draw_tracks(frame, boxes, tids, cids, CLASSES, confs))
    return out_frames

if demo_video:
    for tracker in TRACKERS:
        tname   = tracker_name(tracker)
        best_hp = best_configs.get(tracker, {})
        frames  = annotated_grid_rtdetr(demo_video, FINAL_RTDETR_WEIGHTS,
                                         tracker, best_hp, n_frames=12)
        cols  = 4; rows_n = math.ceil(len(frames) / cols)
        fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 4, rows_n * 3))
        axes = axes.flatten()
        for ax, fr in zip(axes, frames):
            ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); ax.axis('off')
        for ax in axes[len(frames):]: ax.axis('off')
        plt.suptitle(f'RT-DETR+{tname} — annotated frames  ({Path(demo_video).name})',
                     fontsize=12)
        plt.tight_layout()
        out_path = TRACKER_RUNS / f'annotated_{tname}.png'
        plt.savefig(out_path, dpi=120, bbox_inches='tight'); plt.show()
        print(f'Saved: {out_path}')
else:
    print('No eval video found — skipping frame snapshots.')

## 7 — Trajectory Plot

Centroid paths for the best-MOTA tracker on the demo video, overlaid on the first frame.

In [ ]:
best_tracker_name = max(final_results,
                        key=lambda k: final_results[k]['metrics'].get('MOTA') or -1)
best_tracker_yaml = best_tracker_name + '.yaml'
best_hp_traj      = best_configs.get(best_tracker_yaml, {})

def _load_gt_trajectories(video_path, mot_dir):
    seq_name = Path(video_path).stem
    gt_file  = Path(mot_dir) / 'test' / seq_name / 'gt' / 'gt.txt'
    if not gt_file.exists():
        print(f"  [GT] gt.txt not found for {seq_name} — skipping GT overlay")
        return {}
    gt = np.loadtxt(gt_file, delimiter=',')
    if gt.ndim == 1: gt = gt.reshape(1, -1)
    trajs = {}
    for row in gt:
        fn, tid, x, y, w, h = int(row[0]), int(row[1]), row[2], row[3], row[4], row[5]
        cid = int(row[7]) if gt.shape[1] > 7 else 0
        trajs.setdefault(tid, []).append((x + w/2, y + h/2, fn, cid))
    for tid in trajs: trajs[tid].sort(key=lambda p: p[2])
    return trajs

def _draw_trajectory_ax(ax, trajectories, title, bg):
    ax.imshow(cv2.cvtColor(bg, cv2.COLOR_BGR2RGB), alpha=0.35)
    for tid, pts in trajectories.items():
        cid   = pts[0][3]
        color = [c / 255 for c in class_color(cid)]
        xs, ys = [p[0] for p in pts], [p[1] for p in pts]
        ax.plot(xs, ys, color=color, linewidth=1.5, alpha=0.8)
        ax.scatter(xs[-1], ys[-1], color=color, s=25, zorder=5)
        ax.text(xs[-1] + 3, ys[-1] - 3, f'#{tid}', fontsize=6, color=color)
    ax.legend(handles=[mpatches.Patch(color=[c/255 for c in class_color(i)], label=cls)
                       for i, cls in enumerate(CLASSES)],
              loc='upper right', fontsize=7, framealpha=0.7, ncol=2)
    ax.set_title(title, fontsize=11); ax.axis('off')

if demo_video:
    print(f'Trajectory plot — {best_tracker_name}  ({Path(demo_video).name})')
    hp = dict(best_hp_traj)
    run_conf = hp.pop('det_conf', CONF)
    run_iou  = hp.pop('det_iou', IOU)
    t_hp  = {k: v for k, v in hp.items() if k in _TRACKER_KEYS}
    model = _get_rtdetr(str(FINAL_RTDETR_WEIGHTS))
    trk   = _make_tracker(best_tracker_yaml, t_hp)
    pred_trajectories = {}; fi = 0
    cap = cv2.VideoCapture(str(demo_video))
    while True:
        ret, frame = cap.read()
        if not ret: break
        fi += 1
        dets   = _detect_rtdetr(frame, model, conf=run_conf, iou=run_iou)
        tracks = trk.update(dets, frame)
        for t in tracks:
            tid = int(t[4]); cid = int(t[6]) if len(t) > 6 else 0
            cx = (t[0] + t[2]) / 2; cy = (t[1] + t[3]) / 2
            pred_trajectories.setdefault(tid, []).append((cx, cy, fi, cid))
    cap.release()

    gt_trajectories = _load_gt_trajectories(demo_video, MOT_DIR)
    cap2 = cv2.VideoCapture(str(demo_video)); ret, bg = cap2.read(); cap2.release()
    if not ret: bg = np.zeros((720, 1280, 3), dtype=np.uint8)

    n_cols = 2 if gt_trajectories else 1
    fig, axes = plt.subplots(1, n_cols, figsize=(12 * n_cols, 7))
    if n_cols == 1: axes = [axes]
    _draw_trajectory_ax(axes[0], pred_trajectories,
                         f'Predicted — RT-DETR+{best_tracker_name}  ({Path(demo_video).name})', bg)
    if gt_trajectories:
        _draw_trajectory_ax(axes[1], gt_trajectories,
                             f'Ground Truth  ({Path(demo_video).name})', bg)
    plt.tight_layout()
    out_path = TRACKER_RUNS / 'trajectories.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {out_path}')
else:
    print('No eval video found — skipping trajectory plot.')